# 🗄️ SQL Query Optimization — GRPO Training

**OpenEnv Hackathon Grand Finale 2026 — Meta PyTorch × Scaler**

This notebook trains a small LLM (Qwen2.5-0.5B-Instruct) to optimize SQL queries using **Group Relative Policy Optimization (GRPO)**.

The reward signal comes from **real DuckDB query execution** — not keyword matching.

### What you'll see:
- The environment executing real SQL against a 1.5M-row DuckDB database
- GRPO training with live reward curves
- Before vs After comparison of the model's SQL optimization quality

**Runtime:** ~20-30 min on a free Colab T4 GPU

In [ ]:
# @title ⚙️ Step 1: Install dependencies
!pip install -q duckdb fastapi uvicorn pydantic openai trl transformers torch matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 15.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 23.0 MB/s eta 0:00:00:00:0100:01


In [ ]:
# @title 📦 Step 2: Clone the environment
!git clone https://github.com/OfficialAbhinavSingh/SQL-Query-Optimization-Environment- /content/sql-env
%cd /content/sql-env
import sys
sys.path.insert(0, '/content/sql-env')
print('✅ Environment cloned and path set')

Cloning into '/content/sql-env'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 57 (delta 16), reused 54 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 144.69 KiB | 2.45 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/sql-env
✅ Environment cloned and path set


In [ ]:
# @title 🗄️ Step 3: Verify DuckDB environment works
from env import SQLOptimEnv
from models import Action

print('Initialising SQLOptimEnv (DuckDB warm-up ~3s)...')
env = SQLOptimEnv()
obs = env.reset(task_id='task_1_basic_antipatterns')
print(f'✅ Environment ready!')
print(f'   Task: {obs.task_name}')
print(f'   Difficulty: {obs.difficulty}')
print(f'   Max steps: {obs.max_steps}')
print(f'\nSQL to optimize:\n{obs.sql_query}')

Initialising SQLOptimEnv (DuckDB warm-up ~3s)...
✅ Environment ready!
   Task: Basic SQL Anti-pattern Detection
   Difficulty: easy
   Max steps: 3

SQL to optimize:
SELECT *
FROM orders
WHERE CAST(customer_id AS VARCHAR) = '5000'
  AND year(created_at) = 2024;


In [ ]:
# @title 📊 Step 4: Run fallback baseline (no LLM needed)
# This establishes our lower-bound to compare against after training

from baseline_runner import run_fallback_policy, print_comparison_table, TASK_IDS

print('Running deterministic fallback policy...')
fallback_results = run_fallback_policy(env)
print_comparison_table(fallback_results, None)

fb_avg = sum(r['score'] for r in fallback_results.values()) / len(fallback_results)
print(f'Fallback baseline average score: {fb_avg:.4f}')
print('This is our training target — the model should exceed this after GRPO.')

Running deterministic fallback policy...
  [Fallback] easy         | score=0.6800 | speedup=7.53x | correct=False
  [Fallback] medium       | score=0.6500 | speedup=0.59x | correct=True
  [Fallback] medium-hard  | score=0.6800 | speedup=5.55x | correct=False
  [Fallback] hard         | score=0.6500 | speedup=0.77x | correct=True


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  [Fallback] expert       | score=0.6800 | speedup=3.74x | correct=False

  BASELINE RESULTS — SQL Query Optimization Environment
Task                                     Diff          F-Score  F-Spdup  F-Corr
--------------------------------------------------------------------------------
Basic SQL Anti-pattern Detection         easy           0.6800    7.53x      NO
N+1 Correlated Subquery Elimination      medium         0.6500    0.59x     YES
Wildcard LIKE & Projection Optimizatio   medium-hard    0.6800    5.55x      NO
Implicit Cross Join & Scalar Subquery    hard           0.6500    0.77x     YES
Window Function & Full-Scan Audit        expert         0.6800    3.74x      NO
  Fallback avg score : 0.6680

Fallback baseline average score: 0.6680
This is our training target — the model should exceed this after GRPO.


In [ ]:
# @title 🤖 Step 5: Load model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'  # Small model — fits on T4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto' if device == 'cuda' else None,
)
params = sum(p.numel() for p in model.parameters())
print(f'✅ Model loaded — {params:,} parameters')

In [ ]:
# @title 🏋️ Step 6: GRPO Training Loop
import json, random, time, os
import matplotlib.pyplot as plt
import numpy as np
from models import Action
from tasks import TASKS

SYSTEM_PROMPT = """You are an expert database engineer specializing in SQL performance optimization.
Respond ONLY with valid JSON (no markdown):
{"suggestions": [{"issue_type": "...", "line": 1, "description": "...", "severity": "critical", "fix": "..."}],
 "optimized_query": "<complete executable SQL>",
 "summary": "2-4 sentence analysis",
 "estimated_improvement": "e.g. 10x faster",
 "approved": false}"""

def build_prompt(obs):
    return f"Task: {obs.task_name}\nDifficulty: {obs.difficulty}\n\nSchema:\n{obs.schema_info}\n\nSQL to optimize:\n{obs.sql_query}\n\nAnalyze and rewrite this SQL."

def parse_action(text):
    try:
        clean = text.strip()
        if '```' in clean:
            parts = clean.split('```')
            for p in parts:
                p = p.strip().lstrip('json').strip()
                try: return json.loads(p)
                except: continue
        return json.loads(clean)
    except:
        return {"suggestions": [], "optimized_query": "", "summary": "parse error", "estimated_improvement": "unknown", "approved": False}

def compute_advantages(rewards):
    mean_r = sum(rewards) / max(len(rewards), 1)
    std_r = (sum((r - mean_r)**2 for r in rewards) / max(len(rewards), 1))**0.5
    return [(r - mean_r) / (std_r + 1e-8) for r in rewards]

# ── Config ────────────────────────────────────────────────────────
NUM_EPISODES = 100       # reduce to 50 for a quick demo
GROUP_SIZE   = 4         # completions per prompt
MAX_TOKENS   = 512
TEMPERATURE  = 0.8
LR           = 1e-5
LOG_EVERY    = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
task_ids  = list(TASKS.keys())

episode_rewards, episode_losses = [], []
t_start = time.time()

print(f'Starting GRPO training: {NUM_EPISODES} episodes, group size {GROUP_SIZE}')
print(f'Device: {device}  |  Tasks: {len(task_ids)}')
print('=' * 60)

for episode in range(1, NUM_EPISODES + 1):
    task_id = random.choice(task_ids)
    obs = env.reset(task_id=task_id)

    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_prompt(obs)}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors='pt', truncation=True, max_length=1024).to(device)
    prompt_len = inputs['input_ids'].shape[1]

    # Generate G completions
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_TOKENS, temperature=TEMPERATURE,
            do_sample=True, num_return_sequences=GROUP_SIZE,
            pad_token_id=tokenizer.eos_token_id
        )
    completions = [tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in outputs]

    # Score each completion
    rewards = []
    for comp in completions:
        parsed = parse_action(comp)
        action = Action(
            suggestions=parsed.get('suggestions', []),
            optimized_query=parsed.get('optimized_query', ''),
            summary=parsed.get('summary', ''),
            estimated_improvement=parsed.get('estimated_improvement', ''),
            approved=parsed.get('approved', False)
        )
        try:
            env.reset(task_id=task_id)
            result = env.step(action)
            rewards.append(result.reward.score)
        except:
            rewards.append(0.0)

    # GRPO update
    advantages = compute_advantages(rewards)
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    for comp, adv in zip(completions, advantages):
        full_text = chat_text + comp
        inp = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=2048).to(device)
        labels = inp['input_ids'].clone()
        labels[0, :prompt_len] = -100
        out = model(**inp, labels=labels)
        loss = out.loss * adv
        loss.backward()
        total_loss += loss.item()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    mean_r = sum(rewards) / len(rewards)
    episode_rewards.append(mean_r)
    episode_losses.append(total_loss / GROUP_SIZE)

    if episode % LOG_EVERY == 0:
        recent = sum(episode_rewards[-LOG_EVERY:]) / LOG_EVERY
        print(f'Ep {episode:3d}/{NUM_EPISODES} | task={task_id[:25]:<25} | rewards={[f"{r:.3f}" for r in rewards]} | mean={mean_r:.4f} | recent_avg={recent:.4f}')

print('\n✅ Training complete!')

In [ ]:
# @title 📈 Step 7: Plot training curves
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('SQL Query Optimization — GRPO Training Progress', fontsize=14, fontweight='bold')

eps = list(range(1, len(episode_rewards) + 1))
window = max(1, len(episode_rewards) // 10)
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')

ax1.plot(eps, episode_rewards, alpha=0.3, color='#4A90D9', label='Raw reward per episode')
ax1.plot(list(range(window, len(episode_rewards)+1)), smoothed, color='#E74C3C', linewidth=2, label=f'Smoothed (window={window})')
ax1.axhline(y=sum(r['score'] for r in fallback_results.values())/len(fallback_results), 
            color='green', linestyle='--', label=f'Fallback baseline ({fb_avg:.4f})')
ax1.set_xlabel('Training Episode')
ax1.set_ylabel('Mean Group Reward')
ax1.set_title('Reward Curve — Model learns to write faster SQL (higher = better)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1.0)

ax2.plot(eps, episode_losses, alpha=0.4, color='#2ECC71', label='GRPO loss')
smooth_loss = np.convolve(episode_losses, np.ones(window)/window, mode='valid')
ax2.plot(list(range(window, len(episode_losses)+1)), smooth_loss, color='#8E44AD', linewidth=2, label='Smoothed loss')
ax2.set_xlabel('Training Episode')
ax2.set_ylabel('GRPO Policy Loss')
ax2.set_title('Policy Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('/content/sql-env/results', exist_ok=True)
plt.savefig('/content/sql-env/results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Curves saved to results/training_curves.png')

In [ ]:
# @title 🆚 Step 8: Before vs After — Evaluate trained model
from tasks import TASKS

model.eval()
trained_scores = {}

for task_id in TASK_IDS:
    obs = env.reset(task_id=task_id)
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_prompt(obs)}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors='pt', truncation=True, max_length=1024).to(device)
    prompt_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_TOKENS, temperature=0.1,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    parsed = parse_action(completion)
    action = Action(
        suggestions=parsed.get('suggestions', []),
        optimized_query=parsed.get('optimized_query', ''),
        summary=parsed.get('summary', ''),
        estimated_improvement=parsed.get('estimated_improvement', ''),
        approved=parsed.get('approved', False)
    )
    env.reset(task_id=task_id)
    result = env.step(action)
    exec_info = result.info.get('execution') or {}
    trained_scores[task_id] = {
        'task_name': obs.task_name, 'difficulty': obs.difficulty,
        'score': result.reward.score, 'speedup': exec_info.get('speedup', 1.0),
        'correct': exec_info.get('results_match', False)
    }

# Print comparison
print('\n' + '='*70)
print('  BEFORE vs AFTER GRPO TRAINING')
print('='*70)
print(f'{"Task":<30} {"Fallback":>10} {"Trained":>10} {"Delta":>8}')
print('-'*70)
for tid in TASK_IDS:
    fb = fallback_results[tid]['score']
    tr = trained_scores[tid]['score']
    delta = tr - fb
    print(f'{trained_scores[tid]["difficulty"]:<13}{trained_scores[tid]["task_name"][:16]:<17} {fb:>10.4f} {tr:>10.4f} {"+" if delta>=0 else ""}{delta:>7.4f}')
print('='*70)
fb_final = sum(r['score'] for r in fallback_results.values())/len(fallback_results)
tr_final = sum(r['score'] for r in trained_scores.values())/len(trained_scores)
print(f'{"AVERAGE":<30} {fb_final:>10.4f} {tr_final:>10.4f} {"+" if tr_final>=fb_final else ""}{tr_final-fb_final:>7.4f}')
print('='*70)

In [ ]:
# @title 💾 Step 9: Save trained model
import json

os.makedirs('/content/sql-env/checkpoints/final', exist_ok=True)
model.save_pretrained('/content/sql-env/checkpoints/final')
tokenizer.save_pretrained('/content/sql-env/checkpoints/final')

history = {
    'episode_rewards': episode_rewards,
    'episode_losses': episode_losses,
    'fallback_avg': fb_final,
    'trained_avg': tr_final,
    'improvement': tr_final - fb_final,
    'config': {'model': MODEL_NAME, 'episodes': NUM_EPISODES, 'group_size': GROUP_SIZE}
}
with open('/content/sql-env/results/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('✅ Model saved to /content/sql-env/checkpoints/final')
print(f'📊 Training history saved to results/training_history.json')
print(f'\n🏆 Final result: {fb_final:.4f} → {tr_final:.4f} (+{tr_final-fb_final:.4f})')